<a href="https://colab.research.google.com/github/gabriela0402/pathfolio-AI/blob/main/notebooks/00_environment_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pathfolio — Verificação e ingestão inicial

Este notebook será usado para testar a leitura de textos digitados,
arquivos PDF e arquivos DOCX.


## 1. Preparação do ambiente

Nesta etapa, o Pathfolio receberá informações escritas manualmente e as transformará em uma estrutura padronizada.


In [ ]:
from datetime import datetime, timezone

print("Ambiente do Pathfolio funcionando!")


Ambiente do Pathfolio funcionando!


In [ ]:
def create_text_source(
    content: str,
    source_name: str,
    category: str = "manual_text"
) -> dict:
    """
    Converte um texto digitado pelo usuário em uma fonte padronizada.
    """

    cleaned_content = content.strip()

    if not cleaned_content:
        raise ValueError("O conteúdo não pode estar vazio.")

    return {
        "source_type": "manual_text",
        "source_name": source_name,
        "category": category,
        "content": cleaned_content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": None
        }
    }


In [ ]:
manual_source = create_text_source(
    content="""
    Participei de uma iniciação científica envolvendo a comparação de modelos
    de inteligência artificial aplicados à correção de redações. Trabalhei com
    uma base de dados extensa, análise de resultados e comparação de métricas.
    """,
    source_name="Experiência acadêmica",
    category="academic_experience"
)

manual_source


{'source_type': 'manual_text',
 'source_name': 'Experiência acadêmica',
 'category': 'academic_experience',
 'content': 'Participei de uma iniciação científica envolvendo a comparação de modelos\n    de inteligência artificial aplicados à correção de redações. Trabalhei com\n    uma base de dados extensa, análise de resultados e comparação de métricas.',
 'metadata': {'created_at': '2026-11-05T19:26:17.531183+00:00',
  'page': None,
  'file_name': None}}

In [ ]:
assert manual_source["source_type"] == "manual_text"
assert manual_source["source_name"] == "Experiência acadêmica"
assert manual_source["category"] == "academic_experience"
assert len(manual_source["content"]) > 0
assert manual_source["metadata"]["page"] is None

print("Teste de texto manual aprovado.")


Teste de texto manual aprovado.


In [ ]:
try:
    create_text_source(
        content="   ",
        source_name="Texto vazio"
    )
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: O conteúdo não pode estar vazio.


## 2. Leitura de documentos

Nesta etapa, o Pathfolio será capaz de ler arquivos PDF e DOCX.

Cada documento será convertido para uma ou mais fontes padronizadas,
preservando o nome do arquivo e, quando possível, a página de origem.


In [6]:
!pip -q install pypdf python-docx


In [7]:
from pathlib import Path

from pypdf import PdfReader
from docx import Document
from google.colab import files


In [8]:
def read_pdf(file_path: str) -> list[dict]:
    """
    Extrai o texto de um PDF página por página.
    """

    reader = PdfReader(file_path)
    sources = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()

        if not text:
            continue

        sources.append({
            "source_type": "pdf",
            "source_name": Path(file_path).name,
            "category": "uploaded_document",
            "content": text,
            "metadata": {
                "created_at": datetime.now(timezone.utc).isoformat(),
                "page": page_number,
                "file_name": Path(file_path).name
            }
        })

    return sources


In [9]:
def read_docx(file_path: str) -> list[dict]:
    """
    Extrai o texto dos parágrafos de um arquivo DOCX.
    """

    document = Document(file_path)
    paragraphs = []

    for paragraph in document.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    content = "\n".join(paragraphs).strip()

    if not content:
        return []

    return [{
        "source_type": "docx",
        "source_name": Path(file_path).name,
        "category": "uploaded_document",
        "content": content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": Path(file_path).name
        }
    }]


In [10]:
def read_document(file_path: str) -> list[dict]:
    """
    Lê um PDF ou DOCX usando o leitor correspondente.
    """

    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return read_pdf(file_path)

    if extension == ".docx":
        return read_docx(file_path)

    raise ValueError(
        f"Formato não suportado: {extension}. "
        "Envie um arquivo PDF ou DOCX."
    )


In [12]:
uploaded_files = files.upload()


Saving 03_descricao_projeto_ficticio.docx to 03_descricao_projeto_ficticio.docx
Saving 02_relatorio_projeto_ficticio.pdf to 02_relatorio_projeto_ficticio.pdf
Saving 01_curriculo_candidato_ficticio.pdf to 01_curriculo_candidato_ficticio.pdf


In [13]:
uploaded_sources = []

for file_name in uploaded_files.keys():
    document_sources = read_document(file_name)
    uploaded_sources.extend(document_sources)

print(f"Fontes extraídas: {len(uploaded_sources)}")

for source in uploaded_sources:
    print(f"Arquivo: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Página: {source['metadata']['page']}")
    print(f"Caracteres: {len(source['content'])}")
    print("-" * 50)


Fontes extraídas: 4
Arquivo: 03_descricao_projeto_ficticio.docx
Tipo: docx
Página: None
Caracteres: 807
--------------------------------------------------
Arquivo: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Página: 1
Caracteres: 870
--------------------------------------------------
Arquivo: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 1
Caracteres: 907
--------------------------------------------------
Arquivo: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 2
Caracteres: 522
--------------------------------------------------


In [14]:
all_sources = [manual_source] + uploaded_sources

print(f"Total de fontes: {len(all_sources)}")

for index, source in enumerate(all_sources, start=1):
    print(
        f"{index}. "
        f"{source['source_name']} | "
        f"{source['source_type']} | "
        f"{len(source['content'])} caracteres"
    )


Total de fontes: 5
1. Experiência acadêmica | manual_text | 231 caracteres
2. 03_descricao_projeto_ficticio.docx | docx | 807 caracteres
3. 02_relatorio_projeto_ficticio.pdf | pdf | 870 caracteres
4. 01_curriculo_candidato_ficticio.pdf | pdf | 907 caracteres
5. 01_curriculo_candidato_ficticio.pdf | pdf | 522 caracteres


In [15]:
for source in all_sources:
    preview = source["content"][:300].replace("\n", " ")

    print(f"Fonte: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Prévia: {preview}")
    print("=" * 80)


Fonte: Experiência acadêmica
Tipo: manual_text
Prévia: Participei de uma iniciação científica envolvendo a comparação de modelos     de inteligência artificial aplicados à correção de redações. Trabalhei com     uma base de dados extensa, análise de resultados e comparação de métricas.
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Prévia: Descrição de Projeto — Documento de Teste Projeto: Painel de acompanhamento de estudos Contexto Aplicação web fictícia criada para organizar metas de estudo, registrar sessões e sugerir revisões com base no histórico do usuário. Contribuições realizadas Levantamento dos requisitos principais da apli
Fonte: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Prévia: Relatório de Projeto — Documento de Teste Projeto: Classificação de feedbacks acadêmicos Responsável: Marina Oliveira Resumo O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks acadêmicos por tema e prioridade, facilitando a análise por professores. A

In [16]:
assert len(all_sources) >= 1

for source in all_sources:
    assert source["content"].strip()
    assert source["source_name"]
    assert source["source_type"] in {"manual_text", "pdf", "docx"}
    assert "metadata" in source
    assert "file_name" in source["metadata"]

print("Testes de ingestão de documentos aprovados.")


Testes de ingestão de documentos aprovados.


In [17]:
try:
    read_document("arquivo.txt")
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: Formato não suportado: .txt. Envie um arquivo PDF ou DOCX.
